# 02 — Build the GeneRAG Reference Bank

Two artefacts are produced here, both consumed at GeneRAG inference time:

1. **Per-spot foundation-model embeddings** — for every spot in every training slide we
   crop the H&E patch, pass it through a frozen visual encoder (e.g. UNI, EXAONE Path 2.5),
   and store the 7 transpose-augmented embeddings as `.pt`. Together with the spot's
   ground-truth gene profile this forms the Reference Bank `D = {(f^{(i)}_img, y^{(i)}_full)}`.
2. **Anchor gene panel** — a compact gene list that defines the *biological retrieval
   query* `y_anchor`. We follow the GeneRAG paper: union of per-slide highly variable
   genes, mitochondrial / ribosomal genes removed, then narrowed to the intersection
   of top-mean and top-variance genes.

**Prerequisites**
* `01_download_data.ipynb` already ran (so `./hest1k_datasets/<organ>/{st,wsis}/` exist).
* `scanpy`, `anndata`, `torch`, `Pillow`, `tqdm`.
* For the encoders: install [`uni`](https://github.com/mahmoodlab/UNI),
  [`EXAONE Path 2.5`](https://huggingface.co/LGAI-EXAONE/EXAONE-Path-2.5),
  and any others you want (`CONCH`, `H-optimus-0`, `Virchow2`, …) per their own
  instructions.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import anndata
import scanpy as sc
from PIL import Image
from tqdm import tqdm
from huggingface_hub import login

Image.MAX_IMAGE_PIXELS = None  # whole-slide images are large

HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    login(token=HF_TOKEN)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
# Cohort to process — keep these in sync with notebook 01.
ORGAN = 'PRAD'
DATA_ROOT = './hest1k_datasets'
data_path = os.path.join(DATA_ROOT, ORGAN)
tif_path = os.path.join(data_path, 'wsis')
st_path = os.path.join(data_path, 'st')
processed_dir = os.path.join(data_path, 'processed_data')
os.makedirs(processed_dir, exist_ok=True)

# Slide list (PRAD example).
slide_ids = [f'MEND{i}' for i in range(139, 163) if i != 155]
print('slides:', len(slide_ids))

## 1) Load slides and intersect gene panels

Different slides occasionally have slightly different `var_names`. We keep only genes that
appear in every slide so downstream concatenation is well-defined.

In [ ]:
adata_lst = []
common_genes = None
for sid in slide_ids:
    adata = anndata.read_h5ad(os.path.join(st_path, sid + '.h5ad'))
    adata_lst.append(adata)
    common_genes = set(adata.var_names) if common_genes is None else common_genes & set(adata.var_names)

common_genes = sorted(common_genes)
for i, adata in enumerate(adata_lst):
    adata_lst[i] = adata[:, common_genes].copy()

print('common genes across slides:', len(common_genes))

## 2) Anchor-gene selection

We compute highly-variable genes (HVG) per slide via Scanpy, take their union, drop
mitochondrial (`MT-`) and ribosomal (`RPS`, `RPL`) genes that dominate by sheer
magnitude, narrow to the intersection of top-mean and top-std rankings, and finally
keep the **top `ANCHOR_K`** of that intersection ranked by per-gene variance. The
paper uses `ANCHOR_K = 200` for Prostate / Kidney and `300` for Breast.

In [ ]:
# --- HVG union per slide --------------------------------------------
union_hvg: set[str] = set()
for sid, adata in zip(slide_ids, adata_lst):
    A = adata.copy()
    sc.pp.filter_cells(A, min_genes=1)
    sc.pp.filter_genes(A, min_cells=1)
    sc.pp.normalize_total(A, inplace=True)
    sc.pp.log1p(A)
    sc.pp.highly_variable_genes(A, n_top_genes=2000)
    union_hvg.update(A.var_names[A.var['highly_variable']].tolist())

# Drop MT / ribosomal genes.
union_hvg = sorted([g for g in union_hvg if not g.startswith(('MT', 'mt', 'RPS', 'RPL'))])
print('HVG union (post-filter):', len(union_hvg))

In [ ]:
# --- Concatenated count matrix on the HVG union ---------------------
blocks = []
for sid, adata in zip(slide_ids, adata_lst):
    X = adata[:, union_hvg].X
    if hasattr(X, 'toarray'):
        X = X.toarray()
    df = pd.DataFrame(
        X,
        columns=union_hvg,
        index=[f'{sid}_{i}' for i in range(adata.shape[0])],
    )
    blocks.append(df)
all_count_df = pd.concat(blocks, axis=0).fillna(0)
print('combined:', all_count_df.shape)

In [ ]:
# --- Intersection of top-mean and top-std → top-K by variance ------
order_mean = all_count_df.mean(axis=0).sort_values(ascending=False).index
order_std  = all_count_df.std(axis=0).sort_values(ascending=False).index

# `N_TOP` controls the size of the (top-mean ∩ top-std) candidate pool;
# `ANCHOR_K` then keeps the top-ranked-by-variance entries from that pool.
N_TOP = 3000
ANCHOR_K = 200   # 200 for Prostate / Kidney, 300 for Breast (per the paper)

candidates = sorted(set(order_mean[:N_TOP]) & set(order_std[:N_TOP]))
variance = all_count_df[candidates].var(axis=0).sort_values(ascending=False)
selected = variance.index[:ANCHOR_K].tolist()
print(f'N_TOP={N_TOP} -> {len(candidates)} candidates -> top-{ANCHOR_K} by variance = {len(selected)} anchors')

In [ ]:
# Persist the anchor gene list and slide list.
gene_list_path = os.path.join(processed_dir, 'selected_gene_list.txt')
slide_list_path = os.path.join(processed_dir, 'all_slide_lst.txt')
with open(gene_list_path, 'w') as f:
    f.writelines(g + '\n' for g in selected)
with open(slide_list_path, 'w') as f:
    f.writelines(s + '\n' for s in slide_ids)
print('wrote', gene_list_path)
print('wrote', slide_list_path)

## 3) Per-spot foundation-model embeddings

For each slide we crop one square patch per spot (size derived from the Visium
`spot_diameter_fullres`), forward it through a frozen encoder, and store the resulting
embedding. Seven transposed views of each patch are stored together as the augmentation
axis — GeneRAG's data loader averages over this axis at use time.

The cell below shows **UNI** (H&E-only) and **EXAONE Path 2.5** (multi-modal H&E +
omics) — the two backbones reported in the GeneRAG paper. Plug any other foundation
model in the same shape (write the single `get_embedding(patch_pil) -> torch.Tensor[d]`
function).

In [ ]:
# --- Encoder helpers -------------------------------------------------
def load_uni(device: str):
    """Returns (encode_fn, embedding_dim). Requires the `uni` package."""
    from uni import get_encoder
    model, transform = get_encoder(enc_name='uni', device=device)

    def encode(patch: Image.Image) -> torch.Tensor:
        x = transform(patch.resize((224, 224), Image.Resampling.LANCZOS)).unsqueeze(0).to(device)
        with torch.inference_mode():
            return model(x).squeeze(0).detach().cpu()  # (1024,)
    return encode, 1024


def load_exaone(device: str, hf_repo: str = 'LGAI-EXAONE/EXAONE-Path-2.5'):
    """Returns (encode_fn, embedding_dim) for EXAONE Path 2.5.

    The model is fetched from the official HuggingFace repo. If the upstream
    interface changes, the only thing to adjust is the two lines marked
    ``# encoder-specific``.
    """
    import timm
    from torchvision import transforms as T

    # encoder-specific — instantiate the backbone via timm + HuggingFace hub.
    model = timm.create_model(f'hf-hub:{hf_repo}', pretrained=True, num_classes=0).to(device).eval()
    cfg = timm.data.resolve_data_config({}, model=model)
    transform = timm.data.create_transform(**cfg)

    # encoder-specific — embedding dimension (read from the model itself).
    embed_dim = model.num_features if hasattr(model, 'num_features') else model(torch.zeros(1, 3, 224, 224, device=device)).shape[-1]

    def encode(patch: Image.Image) -> torch.Tensor:
        x = transform(patch.resize((224, 224), Image.Resampling.LANCZOS)).unsqueeze(0).to(device)
        with torch.inference_mode(), torch.autocast(device_type=device.split(':')[0], dtype=torch.float16):
            return model(x).squeeze(0).float().detach().cpu()
    return encode, embed_dim

In [ ]:
def extract_slide_embeddings(
    image: Image.Image,
    adata: anndata.AnnData,
    encode_fn,
    embed_dim: int,
    n_augment: int = 7,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Per-spot embeddings for one slide.

    Returns ``(emb, emb_aug)`` of shapes ``(n_spots, d)`` and
    ``(n_spots, n_augment, d)``. ``emb_aug`` averaged over the augmentation
    axis is what the GeneRAG data loader consumes.
    """
    spot_diameter = adata.uns['spatial']['ST']['scalefactors']['spot_diameter_fullres']
    radius = max(112, int(spot_diameter // 2))     # minimum 224x224 patch
    xs = adata.obsm['spatial'][:, 0]
    ys = adata.obsm['spatial'][:, 1]

    emb = torch.zeros(len(xs), embed_dim)
    emb_aug = torch.zeros(len(xs), n_augment, embed_dim)

    for i in tqdm(range(len(xs)), leave=False):
        x, y = xs[i], ys[i]
        patch = image.crop((x - radius, y - radius, x + radius, y + radius))
        emb[i] = encode_fn(patch)
        for t in range(n_augment):
            emb_aug[i, t] = encode_fn(patch.transpose(t))
    return emb, emb_aug

In [ ]:
# --- Loop: extract embeddings for every slide ---------------------
# Pick the encoder(s) you want. The output naming convention matches what
# `generag.data.load_bank_embeddings` expects:
#     {processed_dir}/1spot_{backbone}_ebd_aug/{slide_id}_{backbone}_aug.pt
ENCODERS = {
    'uni':    load_uni,
    # 'exaone': load_exaone,
}

for name, loader in ENCODERS.items():
    out_dir_plain = os.path.join(processed_dir, f'1spot_{name}_ebd')
    out_dir_aug   = os.path.join(processed_dir, f'1spot_{name}_ebd_aug')
    os.makedirs(out_dir_plain, exist_ok=True)
    os.makedirs(out_dir_aug, exist_ok=True)

    encode_fn, embed_dim = loader(DEVICE)
    for sid, adata in zip(slide_ids, adata_lst):
        out_path = os.path.join(out_dir_aug, f'{sid}_{name}_aug.pt')
        if os.path.exists(out_path):
            print(f'skip {sid} ({name}): already exists')
            continue
        image = Image.open(os.path.join(tif_path, sid + '.tif'))
        emb, emb_aug = extract_slide_embeddings(image, adata, encode_fn, embed_dim)
        torch.save(emb,     os.path.join(out_dir_plain, f'{sid}_{name}.pt'))
        torch.save(emb_aug, out_path)
        print(f'{name} {sid}: emb={tuple(emb.shape)} emb_aug={tuple(emb_aug.shape)}')

## What you should see now

```
PRAD/processed_data/
├── selected_gene_list.txt          # anchor gene panel (y_anchor)
├── all_slide_lst.txt               # slide ids participating in the bank
├── 1spot_uni_ebd/                  # per-slide (n_spots, 1024) tensors  → optional
└── 1spot_uni_ebd_aug/              # per-slide (n_spots, 7, 1024)       → consumed by GeneRAG
```

Next: `03_linear_probing.ipynb` trains a linear probe on top of these embeddings to
produce the initial gene predictions (`ŷ_init` in the paper, `test_anchor` in the API).